# **1. Khai báo thư viện và Cấu hình**
- Khai báo thư viện  
- Định nghĩa các đường dẫn  
- Cấu trúc cột mong muốn của bộ dữ liệu gộp  
- Tạo thư mục merged (nếu chưa tạo) để chứa file data sau khi gộp

In [1]:
import pandas as pd
import os

# Đường dẫn
DATA_INPUT = "raw" # Thư mục chứa file raw đã crawl
DATA_OUTPUT = "merged" # Thư mục chứa file dã gộp
OUTPUT_FILE = "data.csv" # Tên file data sau khi gộp

# Các file lưu bằng mã hóa utf-8
UTF8_FILE = [
    "data_trunghieu.csv",
    "data_myha.csv"
]

# Các file lưu bằng mã hóa utf-8-sig
UTF8SIG_FILE = [
    "data_phuongchi.csv",
    "data_manhha.csv"
]

# Tổng hợp danh sách file cần xử lý
ALL_FILE = UTF8_FILE + UTF8SIG_FILE

# Lưu các dataframe đã đọc thành công
DF = []

# Cấu trúc cột chính xác của bộ dữ liệu
COLUMNS = [
    'product_id', 'product_name', 'product_url',
    'category_id', 'category_name', 'category_root_name',
    'price', 'original_price', 'discount_rate',
    'quantity_sold', 'rating_average', 'review_count',
    'is_return_policy', 'is_freeship_xtra', 'is_authentic',
    'image_count', 'video_count',
    'is_brand', 'brand_name', 'origin',
    'store_id', 'store_name', 'store_review_count', 'total_follower', 'is_official',
    'cancel_by_seller_rate', 'cancel_by_seller_rate_status',
    'return_rate', 'return_rate_status'
]

# Tạo thư mục merged nếu chưa tạo
if not os.path.exists(DATA_OUTPUT):
    os.makedirs(DATA_OUTPUT)

# **2. Đọc và kiểm tra cấu trúc của từng file**
- Lặp qua từng file và đọc chúng bằng mã hóa phù hợp (utf-8 hoặc utf-8-sig) 
- Kiểm tra tên cột có khớp với cấu trúc cột mong muốn không


In [2]:
for file in ALL_FILE:
    file_path = os.path.join(DATA_INPUT, file)
    
    # Xác định encoding cho file
    if file in UTF8_FILE:
        encoding_type = 'utf-8'
    else:
        encoding_type = 'utf-8-sig' # Tự động loại bỏ BOM

    try:
        # Đọc file với encoding tương ứng
        df = pd.read_csv(file_path, encoding=encoding_type) 
        
        # Kiểm tra tên cột và loại bỏ BOM
        current_cols = set(df.columns.str.replace('\ufeff', '', regex=False)) # Loại bỏ BOM nếu còn sót
        
        # So sánh tập hợp cột hiện tại với tập hợp cột chính xác của bộ dữ liệu
        if current_cols != set(COLUMNS):
            missing_cols = set(COLUMNS) - current_cols
            extra_cols = current_cols - set(COLUMNS)
            
            print(f"LỖI CẤU TRÚC: File {file} ({encoding_type})")
            if missing_cols:
                print(f"Thiếu cột: {missing_cols}")
            if extra_cols:
                print(f"Cột thừa: {extra_cols}")
            continue
        
        DF.append(df)
        print(f"Đã đọc thành công: {file} ({len(df):,} bản ghi) - Encoding: {encoding_type}")

    except FileNotFoundError:
        print(f"LỖI không tìm thấy file {file} tại đường dẫn {file_path}")
    except Exception as e:
        print(f"LỖI đọc dữ liệu từ file {file}: {e}")

LỖI không tìm thấy file data_trunghieu.csv tại đường dẫn raw\data_trunghieu.csv
LỖI không tìm thấy file data_myha.csv tại đường dẫn raw\data_myha.csv
LỖI không tìm thấy file data_phuongchi.csv tại đường dẫn raw\data_phuongchi.csv
LỖI không tìm thấy file data_manhha.csv tại đường dẫn raw\data_manhha.csv


# **3. Gộp dữ liệu, xử lý trùng lặp và lưu dữ liệu**
- Sau khi tất cả các file đã được đọc thành dataframe, tiến hành gộp lại
- Xử lý các bản ghi trùng lặp
- Lưu dữ liệu đã gộp ra file

In [3]:
if not DF: # Nếu không có dataframe nào được đọc thành công thì dừng
    print("\nKhông có DataFrame nào được đọc thành công. Dừng quá trình.")
else:
    # Gộp tất cả các DataFrame đã đọc
    full_df = pd.concat(DF, ignore_index=True) 

    print(f"ĐÃ HOÀN TẤT GỘP DỮ LIỆU")
    print(f"Tổng số bản ghi (bao gồm trùng lặp): {len(full_df):,}")

    # Kiểm tra và xử lý trùng lặp
    initial_rows = len(full_df)
    
    # Xác định các cột để kiểm tra trùng lặp (Giả sử ID_SP là duy nhất)
    full_df.drop_duplicates(subset=['product_id', 'product_name'], keep='first', inplace=True) 
    rows_after_dedup = len(full_df)

    print(f"Số bản ghi trùng lặp đã bị xóa: {initial_rows - rows_after_dedup:,}")
    print(f"Số bản ghi còn lại sau khi xóa trùng lặp: {rows_after_dedup:,}")
    
    # Lưu data
    output_path = os.path.join(DATA_OUTPUT, OUTPUT_FILE)
    full_df.to_csv(output_path, index=False, encoding='utf-8') 
    
    print(f"\nLưu file thành công tại: {output_path}")


Không có DataFrame nào được đọc thành công. Dừng quá trình.
